In [1]:
import pandas as np
import numpy as np
import h5py

import pandas as pd
import polars as pl

from pymer4.models import glmer

In [2]:
# load r_AB from model simulations
systole = h5py.File('systole.jld', 'r')
systole = np.asarray(systole['systole'])
diastole = h5py.File('diastole.jld', 'r')
diastole = np.asarray(diastole['diastole'])

In [3]:
(systole > .5).mean() # percent of trials labeled coherent 

np.float64(0.391)

In [4]:
(diastole > .5).mean()

np.float64(0.462)

In [ ]:
%%capture

def bootstrap_sim(seed = None, n_per_sub = 50):
    rng = np.random.default_rng(seed)
    idx = rng.choice(np.arange(systole.size), systole.size)
    syst = systole[idx] # bootstrap from simulations
    # segment trials into hypothetical "subjects" and avg w/i subs
    syst = (syst.reshape(systole.size//n_per_sub, n_per_sub) > .5) # 1000 / 50 = 20 subs
    dias = diastole[idx]
    dias = (dias.reshape(diastole.size//n_per_sub, n_per_sub) > .5)
    # return paired difference between systole & diastole conditions
    df = []
    for i in range(syst.shape[0]):
        df.append(pd.DataFrame(dict(
            trial_type = 'systole',
            subject = i,
            response = syst[i]
        )))
    for i in range(dias.shape[0]):
        df.append(pd.DataFrame(dict(
            trial_type = 'diastole',
            subject = i,
            response = dias[i]
        )))
    df = pd.concat(df)
    mod = glmer(
        'response ~ trial_type + (1 | subject)', 
        data = pl.DataFrame(df),
        family = 'binomial', link = 'logit'
    )
    mod.set_factors('trial_type')
    mod.fit(summary = False);
    p = mod.result_fit['p_value'][1] # for trial_type coefficient 
    return p < .05 # True if rejected null

# boostrap 1000 samples
from joblib import Parallel, delayed
rejs = Parallel(n_jobs = -1)(delayed(bootstrap_sim)(i) for i in range(1000))
power = np.mean(rejs)

In [6]:
print(power) # estimated power

0.898
